# Clean Code — Additional Examples

New scenarios to demonstrate clean code (no overlap with the slide examples):

- **Clean variable names** (units, domain clarity, positive booleans, geospatial precision)
- **Clean functions** (IO vs logic, purity, early returns, DI, API surface)
- **Clean comments** (why/assumptions/trade-offs/TODOs, no noise)

Use these as additional sections in your course notebook.

## 1) Clean variable names — new examples

Prefer intent-revealing names with **units**, **domain context**, and **positive booleans**. Be specific with collection names and element shapes.

In [ ]:
# Example A: include units and domain in names
# Vague or misleading
timeout = 120
rate = 5

# Clear + units + domain context
request_timeout_seconds = 120
retry_backoff_seconds = 5


In [ ]:
# Example B: positive booleans > negative booleans
# Double negatives are hard to reason about
is_not_archived = False

# Prefer positive assertions
is_active_record = True


In [ ]:
# Example C: geospatial clarity
# Ambiguous short names
x, y = 52.52, 13.405

# Precise and conventional naming
latitude_deg, longitude_deg = 52.52, 13.405


In [ ]:
# Example D: be specific with collections
# What items? What type?
items = [("Ada", 1815), ("Grace", 1906)]

# Describe the collection and element shape
pioneers_by_birthyear: list[tuple[str, int]] = [("Ada", 1815), ("Grace", 1906)]


**Mini-exercise**

Rename:
```python
dt = 30
cfg = {"lvl": 3}
ok = True
# Aim for: polling_interval_seconds, training_config, has_write_permission
```

## 2) Clean functions — new examples

Separate **IO from logic**, prefer **pure functions** when possible, use **early returns**, keep a **narrow public API**, and use **dependency injection** for testability.

In [ ]:
# Example A: separate IO from logic
from dataclasses import dataclass

@dataclass
class User:
    id: int
    name: str

# One function does IO + parsing + validation
def read_users(path: str) -> list[User]:
    rows = open(path).read().splitlines()
    users = []
    for row in rows:
        id_str, name = row.split(",")
        users.append(User(int(id_str), name.strip()))
    return users

# Split responsibilities
def parse_users(csv_text: str) -> list[User]:
    users = []
    for row in csv_text.splitlines():
        user_id, name = row.split(",")
        users.append(User(int(user_id), name.strip()))
    return users

def read_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# Usage (example):
# users = parse_users(read_text("users.csv"))


In [ ]:
# Example B: avoid hidden mutations (return new value)
# Mutates the input list as a side effect
def normalize_in_place(numbers: list[float]) -> None:
    total = sum(numbers) or 1
    for i, n in enumerate(numbers):
        numbers[i] = n / total

# Pure function returns a new list
def normalized(numbers: list[float]) -> list[float]:
    total = sum(numbers) or 1
    return [n / total for n in numbers]


In [ ]:
# Example C: early returns reduce nesting
# Deep nesting
def can_schedule(meeting, user):
    if meeting is not None:
        if not meeting.is_full():
            if user.has_calendar_access:
                return True
    return False

# Early exits
def can_schedule_clean(meeting, user):
    if meeting is None:
        return False
    if meeting.is_full():
        return False
    if not user.has_calendar_access:
        return False
    return True


In [ ]:
# Example D: narrow the public surface; keep helpers private
def render_report(data: list[dict]) -> str:  # public API
    rows = _format_rows(data)
    return _wrap_html(rows)

def _format_rows(data: list[dict]) -> str:  # private helper
    return "".join(f"<tr><td>{d['id']}</td><td>{d['name']}</td></tr>" for d in data)

def _wrap_html(rows_html: str) -> str:  # private helper
    return f"<table>{rows_html}</table>"


In [ ]:
# Example E: dependency injection for testability
from datetime import datetime, timezone

# Reads current time inside (hard to test)
def token_is_fresh(token) -> bool:
    return (datetime.now(tz=timezone.utc) - token.issued_at).seconds < 900

# Inject a time provider
def token_is_fresh_with_clock(token, now_utc) -> bool:
    return (now_utc() - token.issued_at).seconds < 900


**Mini-exercise**

Refactor to remove mutation and clarify responsibilities:
```python
def apply_discount(cart: list[dict], code: str):
    # modifies cart in place
    for item in cart:
        if code == "BLACKFRIDAY":
            item["price"] *= 0.8
    return sum(i["price"] for i in cart)

# Target: a pure function that returns (new_cart, total) 
# and a separate pricing policy function.
```


## 3) Clean comments — new examples

Explain **why**, document **assumptions/invariants**, record **trade-offs**, keep TODOs **short/actionable**, and avoid noise that restates code.

In [ ]:
# Example A: explain WHY, not WHAT
import re

# WHY: Anchor to start/end to avoid partial matches accepting invalid IDs.
ID_PATTERN = re.compile(r"^[A-Z]{3}-\d{4}$")

def is_valid_id(text: str) -> bool:
    return bool(ID_PATTERN.match(text))


In [ ]:
# Example B: document assumptions & invariants
def allocate_slots(capacity: int, requests: list[str]) -> list[str]:
    """
    Assumptions:
    - capacity >= 0
    - requests are unique ids
    Invariant:
    - returned list length <= capacity
    """
    return requests[:capacity]


In [ ]:
# Example C: record a consciously chosen trade-off
def top_k(stream, k: int):
    """
    We use a min-heap (O(n log k)) instead of full sort (O(n log n))
    because n can be very large while k is small.
    """
    import heapq
    heap = []
    for x in stream:
        if len(heap) < k:
            heapq.heappush(heap, x)
        else:
            heapq.heappushpop(heap, x)
    return sorted(heap, reverse=True)


In [ ]:
# Example D: short, actionable TODO
def read_config(path: str) -> dict:
    # TODO: Support YAML; today only JSON is allowed.
    import json, pathlib
    p = pathlib.Path(path)
    return json.loads(p.read_text(encoding="utf-8"))


In [ ]:
# Example E: avoid noisy restatements
# Noise:
# increment the counter by one
counter = 0
counter += 1

# Self-evident code needs no comment
counter += 1


**Mini-exercise**

Replace the comment with code that makes the intent explicit:
```python
# Increase default window size for high-DPI screens
w = 800
h = 600

# Target: expressive names and maybe a function like
# `default_window_size_for_hidpi()` returning a tuple.
```


## Bonus: small kata to tie it together

Refactor this “works but messy” function to apply the principles above:
```python
def fetch_and_count(q, cache, http):
    # q might be None, return 0
    if not q:
        return 0
    # k is key for cache
    k = "q:" + q
    if k in cache:
        d = cache[k]
    else:
        r = http.get("https://api.example.com/search", params={"q": q})
        if r.status_code != 200:
            return 0
        d = r.json()
        cache[k] = d
    # c is count of results
    c = 0
    for x in d.get("results", []):
        if x.get("ok"):
            c += 1
    return c
```

**Goals:**
- Rename variables (`q`, `k`, `d`, `c` → intent-revealing).
- Separate IO (HTTP/cache) from pure logic.
- Early returns + clear error handling.
- Docstring capturing assumptions and behavior.

## Additional Common Clean Code Principles
- **Meaningful Naming:** Use descriptive, intention-revealing names. For example, prefer `proportional_gain` over `pg` or `x1`.
- **Small, Focused Functions:** Each function should do one thing (single responsibility). Avoid combining unrelated tasks (e.g. computing and printing in one function).
- **Comments and Docstrings:** Use comments sparingly and only when they add clarity. Prefer docstrings for functions/classes.
- **DRY (Don't Repeat Yourself):** Factor out repeated logic into functions or loops. Duplicate code makes maintenance hard.
- **Organized Structure:** Group related code into modules or classes (e.g. a `pid.py` for PID utilities), and keep functions short.

In [ ]:
# Bad: function both computes and prints (mixing concerns)
def process_readings(readings):
    avg = sum(readings) / len(readings)
    print(f"Average reading: {avg}")
    return avg  # Mixing output and return

# Good: focused function returns result only
def calculate_average(readings):
    return sum(readings) / len(readings)

## PEP8 Naming Conventions
PEP8 suggests **snake_case** for functions and variables, **CapWords (CamelCase)** for classes, and **UPPER_SNAKE_CASE** for constants.

In [ ]:
# Module: motor_control.py
class MotorController:
    MAX_SPEED = 100  # Constant in all caps

    def __init__(self, initial_speed):
        self.current_speed = initial_speed  # Instance variable in snake_case

    def increase_speed(self, delta_speed):
        self.current_speed += delta_speed

## Python Data Structures
- **Lists:** Ordered, mutable collections. E.g. `readings = [21.5, 22.0, 21.8]`.
- **Tuples:** Ordered, immutable collections. E.g. `pid_gains = (kp, ki, kd)`.
- **Sets:** Unordered, unique elements. E.g. `unique_ids = {'sensorA', 'sensorB'}`.
- **Dictionaries:** Key-value mappings. E.g. `sensor_data = {'temp': 22.5, 'pressure': 1.01}`.

In [ ]:
readings = [0.1, 0.2, 0.15, 0.18]
pid_gains = (1.0, 0.1, 0.01)
unique_ids = {'sensorA', 'sensorB', 'sensorC'}
sensor_data = {'A': 0.1, 'B': 0.2, 'C': 0.15}

## Indexing, Slicing, and Iteration

In [ ]:
readings = [0.1, 0.2, 0.15, 0.18]
print(readings[0])    # first element
print(readings[-1])   # last element
print(readings[1:3])  # [0.2, 0.15]

# Iteration: apply calibration factor
calibrated = [r * 1.1 for r in readings]
print(calibrated)

## Context Managers

In [ ]:
# Using with for file management
with open('output.log', 'w') as log_file:
    log_file.write("Starting simulation...\n")

# Custom context manager example
class SimulationLogger:
    def __init__(self, filename):
        self.filename = filename

    def __enter__(self):
        self.file = open(self.filename, 'w')
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.file.close()

with SimulationLogger('sim.log') as sim_log:
    sim_log.write("Simulation step 1\n")

## Comprehensions and Lambda

In [ ]:
readings = [0.1, 0.2, 0.15]
doubled = [r * 2 for r in readings]
high = [r for r in readings if r > 0.15]
squared_dict = {i: r**2 for i, r in enumerate(readings)}

sensors = {'A': 2.5, 'B': 1.8, 'C': 3.0}
sorted_keys = sorted(sensors.keys(), key=lambda k: sensors[k])
print(sorted_keys)